In [1]:
base_url = "https://www.maxpreps.com/_next/data/1770310387/football"
endpoint = "stat-leaders/offense/passing.json"

# Loop from 2015 to 2025 (The current season is 2025)
for year in range(2015, 2026):
    # Format the season string (e.g., 2024 -> "24-25")
    short_start = str(year)[-2:]
    short_end = str(year + 1)[-2:]
    season_code = f"{short_start}-{short_end}"
    
    # URL Logic
    if year == 2025:
        # 2025 is the "Current" season, so it typically has no year code in the path
        full_url = f"{base_url}/{endpoint}"
    else:
        # Past years need the year code inserted
        full_url = f"{base_url}/{season_code}/{endpoint}"
        
    print(f"Season {year}: {full_url}")


Season 2015: https://www.maxpreps.com/_next/data/1770310387/football/15-16/stat-leaders/offense/passing.json
Season 2016: https://www.maxpreps.com/_next/data/1770310387/football/16-17/stat-leaders/offense/passing.json
Season 2017: https://www.maxpreps.com/_next/data/1770310387/football/17-18/stat-leaders/offense/passing.json
Season 2018: https://www.maxpreps.com/_next/data/1770310387/football/18-19/stat-leaders/offense/passing.json
Season 2019: https://www.maxpreps.com/_next/data/1770310387/football/19-20/stat-leaders/offense/passing.json
Season 2020: https://www.maxpreps.com/_next/data/1770310387/football/20-21/stat-leaders/offense/passing.json
Season 2021: https://www.maxpreps.com/_next/data/1770310387/football/21-22/stat-leaders/offense/passing.json
Season 2022: https://www.maxpreps.com/_next/data/1770310387/football/22-23/stat-leaders/offense/passing.json
Season 2023: https://www.maxpreps.com/_next/data/1770310387/football/23-24/stat-leaders/offense/passing.json
Season 2024: https:

In [9]:
import requests
import pandas as pd
import os
import time
import random
import logging
import json
import re

# ============================================================================
# CONFIGURATION
# ============================================================================

# The "Build ID" is the magic key for MaxPreps JSON API.
# Verified working as of Feb 2026.
BUILD_ID = "1770310387" 

# Base URL for the Next.js Data API
BASE_URL = f"https://www.maxpreps.com/_next/data/{BUILD_ID}/football"

# Output Directory (Exact path requested)
OUTPUT_ROOT = r"X:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\high_school_stats"

# Scrape Range
START_YEAR = 2011
END_YEAR = 2025

# Max pages to scrape per category (50 players/page * 10 pages = 500 players)
MAX_PAGES = 1

# Stat Categories to Scrape - each category needs specific stat type suffix
CATEGORIES = {
    "passing": "stat-leaders/offense/passing/yds",
    "rushing": "stat-leaders/offense/rushing/yds",
    "receiving": "stat-leaders/offense/receiving/yds",
    "sacks": "stat-leaders/defense/sacks/tot-sacks",
    "tackles": "stat-leaders/defense/tackles/tot-tcks",
    "interceptions": "stat-leaders/defense/interceptions/tot-ints",
    "returns": "stat-leaders/special-teams/total-returns/total"
}

# Browser Headers (Critical to avoid 403 Forbidden)
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "x-nextjs-data": "1",
    "Referer": "https://www.maxpreps.com/"
}

# ============================================================================
# SCRAPER CLASS
# ============================================================================

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class MaxPrepsUniversalScraper:
    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update(HEADERS)

    def get_season_code(self, year):
        """Converts 2015 -> '15-16' for URL construction."""
        start = str(year)[-2:]
        end = str(year + 1)[-2:]
        return f"{start}-{end}"

    def construct_url(self, year, category_path):
        """Builds the correct JSON API URL based on year (Modern vs Legacy paths)."""
        if year == 2025:
            # Modern Pattern (Current Season)
            # Ex: .../football/stat-leaders/offense/passing/yds.json
            return f"{BASE_URL}/{category_path}.json"
        else:
            # Legacy Pattern (Archived Seasons)
            # Ex: .../football/11-12/stat-leaders/offense/passing/yds.json
            season_code = self.get_season_code(year)
            
            # Inject season code into the path
            # category_path is 'stat-leaders/offense/passing/yds'
            # We need '11-12/stat-leaders/offense/passing/yds'
            url_path = f"{season_code}/{category_path}.json"
            return f"{BASE_URL}/{url_path}"

    def parse_modern_data(self, data):
        """Extracts data from the 2025 'statLeadersData' structure (List of Dicts)."""
        try:
            return data['pageProps']['statLeadersData']['seasons'][0]['categories'][0]['leaders']
        except (KeyError, IndexError):
            return []

    def parse_legacy_data(self, data):
        """Extracts data from the 2015-2024 'statLeadersListData' structure (List of Lists)."""
        clean_rows = []
        try:
            # The raw data is a list of lists (no keys, just values)
            raw_rows = data['pageProps']['statLeadersListData']['rows']
            
            # The column definitions tell us what the data in the 'stats' array represents
            col_defs = data['pageProps']['statLeadersListData']['columns']
            stat_headers = [c['header'] for c in col_defs]

            for row in raw_rows:
                # Mapping based on legacy JSON analysis:
                # Index 0: FirstName, 1: LastName, 2: CareerLink, 3: Pos, 4: Rank, 6: School, 10: State
                # Index 8: The list of actual stats [Yds, TD, Int, etc.]
                
                # Extract careerid from the career link if it exists
                career_link = row[2] if len(row) > 2 else ""
                career_id = ""
                if career_link:
                    match = re.search(r'careerid=([a-zA-Z0-9_]+)', career_link)
                    if match:
                        career_id = match.group(1)
                
                player = {
                    "rank": row[4],
                    "athleteFirstName": row[0],
                    "athleteLastName": row[1],
                    "athleteName": f"{row[0]} {row[1]}", # Create full name for consistency
                    "careerLink": career_link,  # Full hyperlink with careerid
                    "careerId": career_id,  # Extracted careerid for matching/deduplication
                    "positionCategory": row[3],
                    "schoolName": row[6],
                    "schoolState": row[10],
                    # The first stat in the list is always the main sorting stat (e.g., Passing Yds)
                    "statValue": row[8][0] if row[8] else 0 
                }
                
                # Dynamically map the rest of the stats
                stats_list = row[8]
                for i, header in enumerate(stat_headers):
                    if i < len(stats_list):
                        # Clean header names (optional)
                        key = header.lower().replace(" ", "_").replace("/", "_")
                        player[key] = stats_list[i]
                
                clean_rows.append(player)
                
            return clean_rows
        except (KeyError, IndexError):
            return []

    def save_data(self, all_players, year, category_name):
        """Saves the accumulated data to both JSON and CSV in strict folder paths."""
        
        if not all_players:
            logger.warning(f"No data to save for {year} {category_name}")
            return

        # 1. Define Paths
        # JSON Path: X:\...\JSON\passing\hs_passing_2015.json
        json_dir = os.path.join(OUTPUT_ROOT, "JSON", category_name)
        json_file = os.path.join(json_dir, f"hs_{category_name}_{year}.json")
        
        # CSV Path: X:\...\CSV\passing\hs_passing_2015.csv
        csv_dir = os.path.join(OUTPUT_ROOT, "CSV", category_name)
        csv_file = os.path.join(csv_dir, f"hs_{category_name}_{year}.csv")

        # 2. Create Directories
        os.makedirs(json_dir, exist_ok=True)
        os.makedirs(csv_dir, exist_ok=True)

        # 3. Save JSON (Raw Dump)
        try:
            with open(json_file, 'w', encoding='utf-8') as f:
                json.dump(all_players, f, indent=4)
            logger.info(f"  -> Saved JSON: {json_file}")
        except Exception as e:
            logger.error(f"Failed to save JSON: {e}")

        # 4. Save CSV (Flattened)
        try:
            df = pd.DataFrame(all_players)
            
            # Normalize columns if needed (e.g., ensure 'year' exists)
            df['year'] = year
            df['category'] = category_name
            
            df.to_csv(csv_file, index=False, encoding='utf-8')
            logger.info(f"  -> Saved CSV:  {csv_file} ({len(df)} records)")
        except Exception as e:
            logger.error(f"Failed to save CSV: {e}")

    def scrape_category(self, year, category_name, category_path):
        """Main logic to scrape all pages for a single category/year."""
        all_players = []
        logger.info(f"\n>>> SCRAPING: YEAR={year} | CATEGORY={category_name.upper()} | PATH={category_path}")

        for page in range(1, MAX_PAGES + 1):
            url = self.construct_url(year, category_path)
            params = {"page": page}
            
            try:
                # Fetch
                r = self.session.get(url, params=params)
                logger.info(f"    ↳ URL: {url} | Status: {r.status_code}")
                
                if r.status_code == 404:
                    logger.info(f"    ↳ Page {page} not found (End of List).")
                    break
                elif r.status_code != 200:
                    logger.warning(f"    ↳ Page {page} failed with status {r.status_code}")
                    break
                
                data = r.json()
                
                # Parse based on structure detection
                props = data.get('pageProps', {})
                batch = []
                
                if 'statLeadersData' in props:
                    # Modern (2025)
                    batch = self.parse_modern_data(data)
                    logger.info(f"    ↳ Format: MODERN (2025 structure)")
                elif 'statLeadersListData' in props:
                    # Legacy (2015-2024)
                    batch = self.parse_legacy_data(data)
                    logger.info(f"    ↳ Format: LEGACY (2015-2024 structure)")
                else:
                    logger.error(f"    ↳ Page {page}: Unknown JSON format. Skipping.")
                    break
                
                if not batch:
                    logger.info(f"    ↳ Page {page} returned 0 players. Stopping.")
                    break
                    
                all_players.extend(batch)
                logger.info(f"    ↳ Page {page}: Retrieved {len(batch)} players | Total so far: {len(all_players)}")
                
                # Be polite (Anti-Rate Limit)
                time.sleep(random.uniform(0.6, 1.2))

            except Exception as e:
                logger.error(f"    ✗ Critical error on Page {page}: {e}")
                break
        
        # Save results after finishing all pages for this year/category
        self.save_data(all_players, year, category_name)
        logger.info(f"✓ {category_name.upper()} {year}: Saved {len(all_players)} total records\n")

    def run(self):
        """Orchestrator - Scrapes all categories for all years 2011-2025."""
        logger.info("\n\n")
        logger.info("=" * 80)
        logger.info("=== STARTING MAXPREPS UNIVERSAL SCRAPER ===")
        logger.info("=" * 80)
        logger.info(f"Output Root: {OUTPUT_ROOT}")
        logger.info(f"Years: {START_YEAR} to {END_YEAR}")
        logger.info(f"Categories: {list(CATEGORIES.keys())}")
        logger.info(f"Target: {(END_YEAR - START_YEAR + 1)} years × {len(CATEGORIES)} categories = {(END_YEAR - START_YEAR + 1) * len(CATEGORIES)} total scrapes")
        logger.info("=" * 80)
        
        total_tasks = (END_YEAR - START_YEAR + 1) * len(CATEGORIES)
        tasks_completed = 0
        
        for year in range(START_YEAR, END_YEAR + 1):
            logger.info(f"\n{'='*80}")
            logger.info(f"YEAR {year} - Scraping all {len(CATEGORIES)} categories")
            logger.info(f"{'='*80}")
            
            for cat_name, cat_path in CATEGORIES.items():
                self.scrape_category(year, cat_name, cat_path)
                tasks_completed += 1
                logger.info(f"[{tasks_completed}/{total_tasks}] Progress: {(tasks_completed/total_tasks)*100:.1f}% complete")
                time.sleep(1)  # Gap between requests
        
        logger.info(f"\n{'='*80}")
        logger.info("✓✓✓ SCRAPING COMPLETE! ✓✓✓")
        logger.info(f"Total tasks completed: {tasks_completed}/{total_tasks}")
        logger.info(f"{'='*80}\n")

if __name__ == "__main__":
    scraper = MaxPrepsUniversalScraper()
    scraper.run()

2026-02-07 11:49:01,041 - INFO - 


2026-02-07 11:49:01,042 - INFO - ================================================================================
2026-02-07 11:49:01,043 - INFO - === STARTING MAXPREPS UNIVERSAL SCRAPER ===
2026-02-07 11:49:01,043 - INFO - ================================================================================
2026-02-07 11:49:01,044 - INFO - Output Root: X:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\high_school_stats
2026-02-07 11:49:01,044 - INFO - Years: 2011 to 2025
2026-02-07 11:49:01,045 - INFO - Categories: ['passing', 'rushing', 'receiving', 'sacks', 'tackles', 'interceptions', 'returns']
2026-02-07 11:49:01,045 - INFO - Target: 15 years × 7 categories = 105 total scrapes


2026-02-07 11:49:01,045 - INFO - ================================================================================
2026-02-07 11:49:01,046 - INFO - 
2026-02-07 11:49:01,046 - INFO - YEAR 2011 - Scraping all 7 categories
2026-02-07 11:49:01,046 - INFO - ================================================================================
2026-02-07 11:49:01,047 - INFO - 
>>> SCRAPING: YEAR=2011 | CATEGORY=PASSING | PATH=stat-leaders/offense/passing/yds
2026-02-07 11:49:02,244 - INFO -     ↳ URL: https://www.maxpreps.com/_next/data/1770310387/football/11-12/stat-leaders/offense/passing/yds.json | Status: 200
2026-02-07 11:49:02,249 - INFO -     ↳ Format: LEGACY (2015-2024 structure)
2026-02-07 11:49:02,250 - INFO -     ↳ Page 1: Retrieved 500 players | Total so far: 500
2026-02-07 11:49:03,165 - INFO -   -> Saved JSON: X:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\high_school_stats\JSON\passing\hs_passing_2011.json
2026-02-07 11:49:03,172 - INFO -   -> Saved CSV: 

In [10]:
import os
import re
import pandas as pd
from pathlib import Path

# ============================================================================
# DATA INTEGRITY CHECKER
# ============================================================================

def check_missing_years():
    """Scans JSON/CSV directories and identifies missing years for each stat category."""
    
    expected_years = set(range(START_YEAR, END_YEAR + 1))  # 2011-2025
    
    # Expected categories
    categories = list(CATEGORIES.keys())
    
    # Store results: {category: {found_years: set, missing_years: set}}
    results = {}
    
    print("=" * 80)
    print("CHECKING FOR MISSING YEARS IN SAVED FILES")
    print("=" * 80)
    print(f"Expected Years: {sorted(expected_years)}")
    print(f"Categories: {categories}")
    print()
    
    for category in categories:
        json_dir = os.path.join(OUTPUT_ROOT, "JSON", category)
        csv_dir = os.path.join(OUTPUT_ROOT, "CSV", category)
        
        found_years = set()
        
        # Check JSON files
        if os.path.exists(json_dir):
            for filename in os.listdir(json_dir):
                # Pattern: hs_<category>_<year>.json
                match = re.search(r'_(\d{4})\.json$', filename)
                if match:
                    year = int(match.group(1))
                    found_years.add(year)
        
        # Check CSV files (redundant check, but verifies consistency)
        if os.path.exists(csv_dir):
            for filename in os.listdir(csv_dir):
                # Pattern: hs_<category>_<year>.csv
                match = re.search(r'_(\d{4})\.csv$', filename)
                if match:
                    year = int(match.group(1))
                    found_years.add(year)
        
        missing_years = expected_years - found_years
        
        results[category] = {
            'found': sorted(found_years),
            'missing': sorted(missing_years),
            'count': len(found_years)
        }
    
    # ========================================================================
    # DISPLAY SUMMARY TABLE
    # ========================================================================
    
    print("\n" + "=" * 80)
    print("SUMMARY: YEARS AVAILABLE BY CATEGORY")
    print("=" * 80)
    print()
    
    summary_data = []
    for category in categories:
        result = results[category]
        summary_data.append({
            'Category': category.upper(),
            'Files Found': result['count'],
            'Available Years': ', '.join(map(str, result['found'])) if result['found'] else "NONE",
            'Missing Years': ', '.join(map(str, result['missing'])) if result['missing'] else "NONE"
        })
    
    summary_df = pd.DataFrame(summary_data)
    print(summary_df.to_string(index=False))
    
    # ========================================================================
    # DETAILED REPORT
    # ========================================================================
    
    print("\n" + "=" * 80)
    print("DETAILED REPORT")
    print("=" * 80)
    
    for category in categories:
        result = results[category]
        print(f"\n{category.upper()}")
        print("-" * 40)
        print(f"  Found: {result['count']}/15 years")
        print(f"  Years: {result['found']}")
        
        if result['missing']:
            print(f"  ❌ MISSING: {result['missing']}")
        else:
            print(f"  ✓ COMPLETE - All years present!")
    
    # ========================================================================
    # COMPLETENESS MATRIX
    # ========================================================================
    
    print("\n" + "=" * 80)
    print("COMPLETENESS MATRIX (✓ = Present, ✗ = Missing)")
    print("=" * 80)
    print()
    
    matrix_data = {}
    for year in sorted(expected_years):
        matrix_data[year] = {}
        for category in categories:
            if year in results[category]['found']:
                matrix_data[year][category] = "✓"
            else:
                matrix_data[year][category] = "✗"
    
    matrix_df = pd.DataFrame(matrix_data).T
    matrix_df.index.name = "Year"
    print(matrix_df.to_string())
    
    # ========================================================================
    # STATISTICS
    # ========================================================================
    
    print("\n" + "=" * 80)
    print("OVERALL STATISTICS")
    print("=" * 80)
    
    total_expected = len(expected_years) * len(categories)
    total_found = sum(results[cat]['count'] for cat in categories)
    total_missing = total_expected - total_found
    completion_pct = (total_found / total_expected) * 100 if total_expected > 0 else 0
    
    print(f"Total Files Expected: {total_expected}")
    print(f"Total Files Found: {total_found}")
    print(f"Total Files Missing: {total_missing}")
    print(f"Completion Rate: {completion_pct:.1f}%")
    
    return results

# Run the check
check_results = check_missing_years()

CHECKING FOR MISSING YEARS IN SAVED FILES
Expected Years: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Categories: ['passing', 'rushing', 'receiving', 'sacks', 'tackles', 'interceptions', 'returns']


SUMMARY: YEARS AVAILABLE BY CATEGORY

     Category  Files Found                                                                          Available Years Missing Years
      PASSING           15 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025          NONE
      RUSHING           15 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025          NONE
    RECEIVING           15 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025          NONE
        SACKS           15 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025          NONE
      TACKLES           15 2011, 2012, 2013, 2014, 2015, 2016, 20

In [11]:
import json
import os
import glob

# ============================================================================
# COLUMN RENAMER - Apply column mappings from column_map.json
# ============================================================================

def rename_columns_from_map():
    """
    Loads column_map.json and renames columns in all JSON and CSV files
    according to the specified mappings.
    """
    
    # 1. Load the column mapping
    column_map_path = os.path.join(OUTPUT_ROOT, "column_map.json")
    
    if not os.path.exists(column_map_path):
        print(f"ERROR: column_map.json not found at {column_map_path}")
        return
    
    with open(column_map_path, 'r', encoding='utf-8') as f:
        column_map = json.load(f)
    
    print("=" * 80)
    print("COLUMN RENAMING UTILITY")
    print("=" * 80)
    print(f"Column Map loaded from: {column_map_path}\n")
    
    # 2. Process each category
    total_files = 0
    total_renamed = 0
    
    for category, rename_dict in column_map.items():
        print(f"\nProcessing {category.upper()}")
        print("-" * 40)
        
        # Get paths for this category
        json_dir = os.path.join(OUTPUT_ROOT, "JSON", category)
        csv_dir = os.path.join(OUTPUT_ROOT, "CSV", category)
        
        # Get all JSON and CSV files for this category
        json_files = glob.glob(os.path.join(json_dir, f"hs_{category}_*.json")) if os.path.exists(json_dir) else []
        csv_files = glob.glob(os.path.join(csv_dir, f"hs_{category}_*.csv")) if os.path.exists(csv_dir) else []
        
        all_files = json_files + csv_files
        
        if not all_files:
            print(f"  No files found for {category}")
            continue
        
        # 3. Process each file
        for file_path in sorted(all_files):
            file_ext = os.path.splitext(file_path)[1]
            filename = os.path.basename(file_path)
            
            try:
                if file_ext == '.json':
                    # ===== Process JSON =====
                    with open(file_path, 'r', encoding='utf-8') as f:
                        data = json.load(f)
                    
                    # Rename keys in each record
                    renamed_data = []
                    for record in data:
                        renamed_record = {}
                        for old_key, value in record.items():
                            new_key = rename_dict.get(old_key, old_key)
                            renamed_record[new_key] = value
                        renamed_data.append(renamed_record)
                    
                    # Write back
                    with open(file_path, 'w', encoding='utf-8') as f:
                        json.dump(renamed_data, f, indent=4)
                    
                    print(f"  ✓ JSON: {filename} ({len(renamed_data)} records)")
                    total_renamed += 1
                
                elif file_ext == '.csv':
                    # ===== Process CSV =====
                    df = pd.read_csv(file_path, encoding='utf-8')
                    
                    # Rename columns using the mapping
                    df = df.rename(columns=rename_dict)
                    
                    # Write back
                    df.to_csv(file_path, index=False, encoding='utf-8')
                    
                    print(f"  ✓ CSV:  {filename} ({len(df)} records, {len(df.columns)} columns)")
                    total_renamed += 1
                
                total_files += 1
                
            except Exception as e:
                print(f"  ✗ ERROR processing {filename}: {e}")
        
        print(f"  Summary: {len(json_files)} JSON + {len(csv_files)} CSV files processed")
    
    # 4. Final summary
    print("\n" + "=" * 80)
    print("COLUMN RENAMING COMPLETE")
    print("=" * 80)
    print(f"Total files processed: {total_files}")
    print(f"Total files renamed: {total_renamed}")
    print("\nAll columns have been renamed according to column_map.json")

# Run the renamer
rename_columns_from_map()

COLUMN RENAMING UTILITY
Column Map loaded from: X:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\high_school_stats\column_map.json


Processing PASSING
----------------------------------------
  ✓ CSV:  hs_passing_2011.csv (500 records, 21 columns)
  ✓ CSV:  hs_passing_2012.csv (500 records, 21 columns)
  ✓ CSV:  hs_passing_2013.csv (500 records, 21 columns)
  ✓ CSV:  hs_passing_2014.csv (500 records, 21 columns)
  ✓ CSV:  hs_passing_2015.csv (500 records, 21 columns)
  ✓ CSV:  hs_passing_2016.csv (500 records, 21 columns)
  ✓ CSV:  hs_passing_2017.csv (500 records, 21 columns)
  ✓ CSV:  hs_passing_2018.csv (500 records, 21 columns)
  ✓ CSV:  hs_passing_2019.csv (500 records, 21 columns)
  ✓ CSV:  hs_passing_2020.csv (500 records, 21 columns)
  ✓ CSV:  hs_passing_2021.csv (500 records, 21 columns)
  ✓ CSV:  hs_passing_2022.csv (500 records, 21 columns)
  ✓ CSV:  hs_passing_2023.csv (500 records, 21 columns)
  ✓ CSV:  hs_passing_2024.csv (500 records, 21 colum